# Advanced Problems with Solutions: Infinite Iterators

Topic: Python `itertools.count`, `itertools.cycle`, and `itertools.repeat`.

This notebook extends the source lesson on infinite iterators into an advanced practice notebook. Every problem includes a complete solution, runnable examples, and sanity checks.

## How to use this notebook

1. Run the setup cell first.
2. Try each problem before reading the solution.
3. Run the solution cell and inspect the tests.
4. Modify the examples to build intuition about laziness, object identity, and infinite streams.


## Best practices for infinite iterators

- Never call `list()` on an infinite iterator.
- Always bound infinite streams with `islice`, `takewhile`, `break`, or a finite consumer.
- Use `Decimal` or `Fraction` when exact stepping matters.
- Be careful with `cycle`: it saves every item it has seen from the input iterable.
- Be careful with `repeat`: it returns the same object reference every time.
- Prefer small reusable generator functions and include tests for edge cases.


In [1]:
from itertools import (
    count,
    cycle,
    repeat,
    islice,
    chain,
    takewhile,
    starmap,
)
from collections import namedtuple, defaultdict, deque, Counter
from decimal import Decimal, getcontext
from fractions import Fraction
from datetime import datetime, timedelta
from copy import copy, deepcopy
import operator
import math
import random

random.seed(42)
getcontext().prec = 28

def take(n, iterable):
    """Return the first n items from any iterable as a list."""
    if n < 0:
        raise ValueError("n must be non-negative")
    return list(islice(iterable, n))

def nth(iterable, n):
    """Return the zero-based nth item from any iterable."""
    if n < 0:
        raise ValueError("n must be non-negative")
    return next(islice(iterable, n, None))

def assert_equal(actual, expected):
    assert actual == expected, f"Expected {expected!r}, got {actual!r}"

print("Setup complete.")


Setup complete.


# Warm-up examples

These examples review the core behavior of the three infinite iterator tools.


In [2]:
# count: integer stepping
ints = count(10, 3)
assert_equal(take(6, ints), [10, 13, 16, 19, 22, 25])

# count: Decimal stepping for exact decimal arithmetic
decimals = count(Decimal("0.0"), Decimal("0.1"))
assert_equal(take(5, decimals), [
    Decimal("0.0"),
    Decimal("0.1"),
    Decimal("0.2"),
    Decimal("0.3"),
    Decimal("0.4"),
])

# count: Fraction stepping for exact rational arithmetic
fractions = count(Fraction(1, 3), Fraction(1, 6))
assert_equal(take(5, fractions), [
    Fraction(1, 3),
    Fraction(1, 2),
    Fraction(2, 3),
    Fraction(5, 6),
    Fraction(1, 1),
])

print("count examples passed.")


count examples passed.


In [3]:
# cycle: repeat a finite pattern forever, but consume only a bounded prefix
traffic_lights = cycle(["red", "green", "yellow"])
assert_equal(take(10, traffic_lights), [
    "red", "green", "yellow",
    "red", "green", "yellow",
    "red", "green", "yellow",
    "red",
])

# cycle also works with a generator, but it caches the values it consumes
def color_generator():
    yield "cyan"
    yield "magenta"
    yield "yellow"

cycled_generator = cycle(color_generator())
assert_equal(take(8, cycled_generator), [
    "cyan", "magenta", "yellow",
    "cyan", "magenta", "yellow",
    "cyan", "magenta",
])

print("cycle examples passed.")


cycle examples passed.


In [4]:
# repeat: repeat the same object reference
original = [1, 2, 3]
repeated = list(repeat(original, 3))

assert repeated[0] is original
assert repeated[1] is original
assert repeated[2] is original

repeated[0].append(99)
assert_equal(repeated, [[1, 2, 3, 99], [1, 2, 3, 99], [1, 2, 3, 99]])

print("repeat aliasing example passed.")


repeat aliasing example passed.


# Problem 1 — Safe sampling from infinite iterators

Write two utilities:

1. `take(n, iterable)` should return the first `n` items.
2. `nth(iterable, n)` should return the zero-based `n`th item.

Requirements:

- They must work with infinite iterators.
- They must not convert the whole iterable to a list.
- They must reject negative `n`.


In [5]:
# Solution 1

def take(n, iterable):
    """Return the first n items from an iterable without exhausting infinite inputs."""
    if n < 0:
        raise ValueError("n must be non-negative")
    return list(islice(iterable, n))

def nth(iterable, n):
    """Return the zero-based nth item from an iterable."""
    if n < 0:
        raise ValueError("n must be non-negative")
    return next(islice(iterable, n, None))

assert_equal(take(5, count(100, 10)), [100, 110, 120, 130, 140])
assert_equal(nth(count(100, 10), 0), 100)
assert_equal(nth(count(100, 10), 4), 140)

try:
    take(-1, count())
except ValueError as ex:
    assert "non-negative" in str(ex)
else:
    raise AssertionError("Expected ValueError")

print("Problem 1 solution passed.")


Problem 1 solution passed.


# Problem 2 — Finite arithmetic ranges using `count`

Implement `count_until(start, step, stop, inclusive=False)`.

It should lazily produce values from `start`, adding `step`, and stop before crossing `stop`.

Examples:

```python
list(count_until(0, 2, 7))              # [0, 2, 4, 6]
list(count_until(0, 2, 6, True))        # [0, 2, 4, 6]
list(count_until(10, -3, 0))            # [10, 7, 4, 1]
```

Requirements:

- Support positive and negative steps.
- Reject a zero step.
- Work with `int`, `Decimal`, and `Fraction`.


In [6]:
# Solution 2

def count_until(start, step, stop, inclusive=False):
    """Lazy arithmetic range based on itertools.count."""
    if step == 0:
        raise ValueError("step cannot be zero")

    if step > 0:
        keep = (lambda x: x <= stop) if inclusive else (lambda x: x < stop)
    else:
        keep = (lambda x: x >= stop) if inclusive else (lambda x: x > stop)

    yield from takewhile(keep, count(start, step))

assert_equal(list(count_until(0, 2, 7)), [0, 2, 4, 6])
assert_equal(list(count_until(0, 2, 6, inclusive=True)), [0, 2, 4, 6])
assert_equal(list(count_until(10, -3, 0)), [10, 7, 4, 1])

assert_equal(
    list(count_until(Decimal("0.0"), Decimal("0.2"), Decimal("1.0"))),
    [Decimal("0.0"), Decimal("0.2"), Decimal("0.4"), Decimal("0.6"), Decimal("0.8")]
)

assert_equal(
    list(count_until(Fraction(0), Fraction(1, 3), Fraction(1), inclusive=True)),
    [Fraction(0), Fraction(1, 3), Fraction(2, 3), Fraction(1)]
)

try:
    list(count_until(0, 0, 10))
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for zero step")

print("Problem 2 solution passed.")


Problem 2 solution passed.


# Problem 3 — Avoid floating-point drift

Using `count(0.0, 0.1)` can produce floating-point artifacts.

Create three sequences from `0` through `1` by tenths:

1. Using `float`
2. Using `Decimal`
3. Using `Fraction`

Then write a helper `tenths_as_decimal_strings()` that returns exactly:

```python
["0.0", "0.1", "0.2", ..., "1.0"]
```


In [7]:
# Solution 3

float_tenths = take(11, count(0.0, 0.1))
decimal_tenths = take(11, count(Decimal("0.0"), Decimal("0.1")))
fraction_tenths = take(11, count(Fraction(0), Fraction(1, 10)))

# Float values may display artifacts in some positions.
print("float:", float_tenths)
print("Decimal:", decimal_tenths)
print("Fraction:", fraction_tenths)

def tenths_as_decimal_strings():
    return [str(x.quantize(Decimal("0.0"))) for x in take(11, count(Decimal("0.0"), Decimal("0.1")))]

assert_equal(tenths_as_decimal_strings(), [
    "0.0", "0.1", "0.2", "0.3", "0.4", "0.5",
    "0.6", "0.7", "0.8", "0.9", "1.0"
])

print("Problem 3 solution passed.")


float: [0.0, 0.1, 0.2, 0.30000000000000004, 0.4, 0.5, 0.6, 0.7, 0.7999999999999999, 0.8999999999999999, 0.9999999999999999]
Decimal: [Decimal('0.0'), Decimal('0.1'), Decimal('0.2'), Decimal('0.3'), Decimal('0.4'), Decimal('0.5'), Decimal('0.6'), Decimal('0.7'), Decimal('0.8'), Decimal('0.9'), Decimal('1.0')]
Fraction: [Fraction(0, 1), Fraction(1, 10), Fraction(1, 5), Fraction(3, 10), Fraction(2, 5), Fraction(1, 2), Fraction(3, 5), Fraction(7, 10), Fraction(4, 5), Fraction(9, 10), Fraction(1, 1)]
Problem 3 solution passed.


# Problem 4 — Date/time sequences without misusing `itertools.count`

`itertools.count` is for numeric values. A common need is to create an infinite sequence of datetimes.

Implement `datetime_count(start, step)`.

Requirements:

- `start` must be a `datetime`.
- `step` must be a `timedelta`.
- Return an infinite lazy iterator.
- Demonstrate daily, hourly, and weekly schedules.


In [8]:
# Solution 4

def datetime_count(start, step):
    """Yield datetimes forever, starting at start and adding step each time."""
    if not isinstance(start, datetime):
        raise TypeError("start must be a datetime")
    if not isinstance(step, timedelta):
        raise TypeError("step must be a timedelta")
    current = start
    while True:
        yield current
        current += step

start = datetime(2026, 1, 1, 9, 0)

daily = take(3, datetime_count(start, timedelta(days=1)))
hourly = take(4, datetime_count(start, timedelta(hours=6)))
weekly = take(3, datetime_count(start, timedelta(weeks=1)))

assert_equal(daily, [
    datetime(2026, 1, 1, 9, 0),
    datetime(2026, 1, 2, 9, 0),
    datetime(2026, 1, 3, 9, 0),
])

assert_equal(hourly, [
    datetime(2026, 1, 1, 9, 0),
    datetime(2026, 1, 1, 15, 0),
    datetime(2026, 1, 1, 21, 0),
    datetime(2026, 1, 2, 3, 0),
])

assert_equal(weekly[-1], datetime(2026, 1, 15, 9, 0))

print("Problem 4 solution passed.")


Problem 4 solution passed.


# Problem 5 — Custom `enumerate` with arbitrary step

Implement `enumerate_step(iterable, start=0, step=1)` using `count`.

Examples:

```python
list(enumerate_step("abc", start=10, step=5))
# [(10, "a"), (15, "b"), (20, "c")]
```

Requirements:

- Must be lazy.
- Must stop when the input iterable stops.
- Must support non-integer steps such as `Fraction(1, 2)`.


In [9]:
# Solution 5

def enumerate_step(iterable, start=0, step=1):
    """Enumerate an iterable with a custom start and step."""
    return zip(count(start, step), iterable)

assert_equal(
    list(enumerate_step("abc", start=10, step=5)),
    [(10, "a"), (15, "b"), (20, "c")]
)

assert_equal(
    list(enumerate_step(["x", "y", "z"], start=Fraction(1, 2), step=Fraction(1, 4))),
    [(Fraction(1, 2), "x"), (Fraction(3, 4), "y"), (Fraction(1, 1), "z")]
)

# Laziness check: the infinite count is bounded by the finite iterable.
assert_equal(list(enumerate_step([], start=999)), [])

print("Problem 5 solution passed.")


Problem 5 solution passed.


# Problem 6 — Deal a deck using `cycle`

Create a standard 52-card deck and implement `deal(deck, players)`.

Requirements:

- `players` is a list of player names.
- Return a dictionary mapping each player to a list of cards.
- Deal one card at a time in round-robin order.
- Use `cycle`.
- Validate that there is at least one player.


In [10]:
# Solution 6

Card = namedtuple("Card", "rank suit")

def card_deck():
    ranks = tuple(str(n) for n in range(2, 11)) + tuple("JQKA")
    suits = ("Spades", "Hearts", "Diamonds", "Clubs")
    for suit in suits:
        for rank in ranks:
            yield Card(rank, suit)

def deal(deck, players):
    if not players:
        raise ValueError("players must not be empty")
    hands = {player: [] for player in players}
    player_cycle = cycle(players)
    for card in deck:
        player = next(player_cycle)
        hands[player].append(card)
    return hands

players = ["Ada", "Linus", "Guido", "Grace"]
hands = deal(card_deck(), players)

assert_equal(set(hands), set(players))
assert_equal([len(hands[p]) for p in players], [13, 13, 13, 13])
assert_equal(hands["Ada"][0], Card("2", "Spades"))
assert_equal(hands["Linus"][0], Card("3", "Spades"))

try:
    deal(card_deck(), [])
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for empty players")

print("Problem 6 solution passed.")


Problem 6 solution passed.


# Problem 7 — Support rota with dates and `cycle`

Implement `support_rota(engineers, start_date, days)`.

Requirements:

- Return a list of `(date, engineer)` pairs.
- Rotate engineers using `cycle`.
- Generate dates lazily with your `datetime_count`.
- Validate inputs.


In [11]:
# Solution 7

def support_rota(engineers, start_date, days):
    if not engineers:
        raise ValueError("engineers must not be empty")
    if days < 0:
        raise ValueError("days must be non-negative")

    dates = datetime_count(start_date, timedelta(days=1))
    assignments = zip(dates, cycle(engineers))
    return take(days, assignments)

rota = support_rota(
    engineers=["Ada", "Linus", "Guido"],
    start_date=datetime(2026, 7, 1),
    days=8,
)

expected_names = ["Ada", "Linus", "Guido", "Ada", "Linus", "Guido", "Ada", "Linus"]
assert_equal([name for _, name in rota], expected_names)
assert_equal(rota[0][0], datetime(2026, 7, 1))
assert_equal(rota[-1][0], datetime(2026, 7, 8))

print("Problem 7 solution passed.")


Problem 7 solution passed.


# Problem 8 — Weighted infinite cycle

Implement `weighted_cycle(weights)`.

Example:

```python
weights = {"A": 2, "B": 1, "C": 3}
take(10, weighted_cycle(weights))
# ["A", "A", "B", "C", "C", "C", "A", "A", "B", "C"]
```

Requirements:

- Use `repeat` to expand each weighted item.
- Use `cycle` to make the expanded pattern infinite.
- Reject negative weights.
- Reject all-zero weights.


In [12]:
# Solution 8

def weighted_cycle(weights):
    if any(weight < 0 for weight in weights.values()):
        raise ValueError("weights cannot be negative")

    pattern = tuple(chain.from_iterable(
        repeat(item, weight)
        for item, weight in weights.items()
    ))

    if not pattern:
        raise ValueError("at least one weight must be positive")

    return cycle(pattern)

weights = {"A": 2, "B": 1, "C": 3}
assert_equal(
    take(10, weighted_cycle(weights)),
    ["A", "A", "B", "C", "C", "C", "A", "A", "B", "C"]
)

assert_equal(Counter(take(60, weighted_cycle(weights))), Counter({"A": 20, "B": 10, "C": 30}))

try:
    weighted_cycle({"A": 0, "B": 0})
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for all-zero weights")

try:
    weighted_cycle({"A": -1})
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for negative weights")

print("Problem 8 solution passed.")


Problem 8 solution passed.


# Problem 9 — Fix the `repeat` mutable-aliasing bug

`repeat([] , 3)` repeats the same list object three times.

Implement `repeat_factory(factory, times=None)`.

Requirements:

- If `times` is an integer, produce exactly that many fresh objects.
- If `times is None`, produce fresh objects forever.
- Use `repeat` internally.
- Demonstrate that each returned list is independent.


In [13]:
# Solution 9

def repeat_factory(factory, times=None):
    """Call factory repeatedly, yielding a fresh object each time."""
    if times is None:
        triggers = repeat(None)
    else:
        if times < 0:
            raise ValueError("times must be non-negative or None")
        triggers = repeat(None, times)

    for _ in triggers:
        yield factory()

lists = list(repeat_factory(list, 3))
assert_equal(lists, [[], [], []])
assert lists[0] is not lists[1]
assert lists[1] is not lists[2]
assert lists[0] is not lists[2]

lists[0].append("only first")
assert_equal(lists, [["only first"], [], []])

fresh_dicts = take(3, repeat_factory(dict))
fresh_dicts[0]["x"] = 1
assert_equal(fresh_dicts, [{"x": 1}, {}, {}])

print("Problem 9 solution passed.")


Problem 9 solution passed.


# Problem 10 — Sensor event stream

Build an infinite stream of sensor events.

Implement:

```python
sensor_events(device_id, start, seconds, statuses)
```

Each yielded event should be a dictionary:

```python
{
    "sequence": 1,
    "device_id": "sensor-7",
    "timestamp": datetime(...),
    "status": "OK"
}
```

Requirements:

- `sequence` starts at 1 using `count`.
- `device_id` repeats forever using `repeat`.
- `timestamp` uses `datetime_count`.
- `status` cycles through the provided statuses.
- Validate that `statuses` is not empty.


In [14]:
# Solution 10

def sensor_events(device_id, start, seconds, statuses):
    if not statuses:
        raise ValueError("statuses must not be empty")

    timestamps = datetime_count(start, timedelta(seconds=seconds))

    for sequence, repeated_device_id, timestamp, status in zip(
        count(1),
        repeat(device_id),
        timestamps,
        cycle(statuses),
    ):
        yield {
            "sequence": sequence,
            "device_id": repeated_device_id,
            "timestamp": timestamp,
            "status": status,
        }

events = take(5, sensor_events(
    "sensor-7",
    datetime(2026, 1, 1, 0, 0, 0),
    seconds=15,
    statuses=["OK", "WARN", "OK", "FAIL"],
))

assert_equal([event["sequence"] for event in events], [1, 2, 3, 4, 5])
assert_equal([event["device_id"] for event in events], ["sensor-7"] * 5)
assert_equal([event["status"] for event in events], ["OK", "WARN", "OK", "FAIL", "OK"])
assert_equal(events[2]["timestamp"], datetime(2026, 1, 1, 0, 0, 30))

print("Problem 10 solution passed.")


Problem 10 solution passed.


# Problem 11 — Infinite pagination URLs

Implement `page_urls(base_url, start_page=1)`.

Requirements:

- Lazily generate URLs like `https://example.test/items?page=1`.
- Use `count`.
- Support custom `start_page`.
- Demonstrate taking only the first 5 URLs.


In [15]:
# Solution 11

def page_urls(base_url, start_page=1):
    if start_page < 1:
        raise ValueError("start_page must be >= 1")
    for page in count(start_page):
        separator = "&" if "?" in base_url else "?"
        yield f"{base_url}{separator}page={page}"

urls = take(5, page_urls("https://example.test/items"))
assert_equal(urls, [
    "https://example.test/items?page=1",
    "https://example.test/items?page=2",
    "https://example.test/items?page=3",
    "https://example.test/items?page=4",
    "https://example.test/items?page=5",
])

urls_with_query = take(3, page_urls("https://example.test/search?q=python", start_page=4))
assert_equal(urls_with_query, [
    "https://example.test/search?q=python&page=4",
    "https://example.test/search?q=python&page=5",
    "https://example.test/search?q=python&page=6",
])

print("Problem 11 solution passed.")


Problem 11 solution passed.


# Problem 12 — Partition rows round-robin

Implement `partition_rows(rows, number_of_partitions)`.

Requirements:

- Assign rows to partitions `0, 1, ..., number_of_partitions - 1`.
- Use `cycle`.
- Return a dictionary `{partition_id: [rows...]}`.
- Preserve original row order within each partition.


In [16]:
# Solution 12

def partition_rows(rows, number_of_partitions):
    if number_of_partitions <= 0:
        raise ValueError("number_of_partitions must be positive")

    partitions = {i: [] for i in range(number_of_partitions)}

    for partition_id, row in zip(cycle(range(number_of_partitions)), rows):
        partitions[partition_id].append(row)

    return partitions

rows = list("ABCDEFGHIJ")
partitions = partition_rows(rows, 3)

assert_equal(partitions, {
    0: ["A", "D", "G", "J"],
    1: ["B", "E", "H"],
    2: ["C", "F", "I"],
})

print("Problem 12 solution passed.")


Problem 12 solution passed.


# Problem 13 — Bounded cycling helper

Implement `limited_cycle(iterable, total)`.

Requirements:

- Return exactly `total` items by cycling over `iterable`.
- If `total == 0`, return an empty list.
- Reject empty iterables when `total > 0`.
- Materialize the pattern as a tuple so you can check whether it is empty.


In [17]:
# Solution 13

def limited_cycle(iterable, total):
    if total < 0:
        raise ValueError("total must be non-negative")

    pattern = tuple(iterable)

    if total > 0 and not pattern:
        raise ValueError("cannot cycle an empty iterable when total > 0")

    return take(total, cycle(pattern))

assert_equal(limited_cycle(["x", "y"], 7), ["x", "y", "x", "y", "x", "y", "x"])
assert_equal(limited_cycle([], 0), [])

try:
    limited_cycle([], 1)
except ValueError:
    pass
else:
    raise AssertionError("Expected ValueError for empty pattern")

print("Problem 13 solution passed.")


Problem 13 solution passed.


# Problem 14 — Safer cycling with a maximum pattern size

`cycle` caches values from its input iterable. Accidentally cycling a very large iterable can use lots of memory.

Implement `bounded_cycle(iterable, max_pattern_size)`.

Requirements:

- Materialize at most `max_pattern_size + 1` items from the input.
- If the iterable has more than `max_pattern_size` items, raise `ValueError`.
- If the iterable is empty, raise `ValueError`.
- Return `cycle(pattern)` for valid inputs.


In [18]:
# Solution 14

def bounded_cycle(iterable, max_pattern_size):
    if max_pattern_size <= 0:
        raise ValueError("max_pattern_size must be positive")

    sample = take(max_pattern_size + 1, iterable)

    if not sample:
        raise ValueError("cannot cycle an empty iterable")

    if len(sample) > max_pattern_size:
        raise ValueError("pattern is larger than max_pattern_size")

    return cycle(tuple(sample))

small = bounded_cycle(iter(["A", "B", "C"]), max_pattern_size=3)
assert_equal(take(8, small), ["A", "B", "C", "A", "B", "C", "A", "B"])

try:
    bounded_cycle(range(5), max_pattern_size=4)
except ValueError as ex:
    assert "larger" in str(ex)
else:
    raise AssertionError("Expected ValueError for too-large iterable")

print("Problem 14 solution passed.")


Problem 14 solution passed.


# Problem 15 — Round-robin over uneven iterables

The built-in `cycle` is not ideal when the set of active iterators changes because some iterators become exhausted.

Implement `roundrobin(*iterables)`.

Example:

```python
list(roundrobin("ABC", [1, 2], ("x", "y", "z", "w")))
# ["A", 1, "x", "B", 2, "y", "C", "z", "w"]
```

Requirements:

- Continue until all input iterables are exhausted.
- Do not raise `StopIteration` to the caller.
- Preserve round-robin order among active iterators.


In [19]:
# Solution 15

def roundrobin(*iterables):
    iterators = deque(iter(it) for it in iterables)

    while iterators:
        iterator = iterators.popleft()

        try:
            value = next(iterator)
        except StopIteration:
            continue
        else:
            yield value
            iterators.append(iterator)

assert_equal(
    list(roundrobin("ABC", [1, 2], ("x", "y", "z", "w"))),
    ["A", 1, "x", "B", 2, "y", "C", "z", "w"]
)

assert_equal(list(roundrobin([], [1, 2, 3], [])), [1, 2, 3])
assert_equal(list(roundrobin()), [])

print("Problem 15 solution passed.")


Problem 15 solution passed.


# Problem 16 — Rotating dealer button

In many card games, the dealer position rotates after each hand.

Implement `dealer_schedule(players, number_of_hands)`.

Requirements:

- Use `cycle(players)` for dealers.
- Use `count(1)` for hand numbers.
- Return a list of dictionaries with keys `hand_number` and `dealer`.
- Validate inputs.


In [20]:
# Solution 16

def dealer_schedule(players, number_of_hands):
    if not players:
        raise ValueError("players must not be empty")
    if number_of_hands < 0:
        raise ValueError("number_of_hands must be non-negative")

    return [
        {"hand_number": hand_number, "dealer": dealer}
        for hand_number, dealer in islice(zip(count(1), cycle(players)), number_of_hands)
    ]

schedule = dealer_schedule(["Ada", "Linus", "Guido"], 8)
assert_equal(schedule, [
    {"hand_number": 1, "dealer": "Ada"},
    {"hand_number": 2, "dealer": "Linus"},
    {"hand_number": 3, "dealer": "Guido"},
    {"hand_number": 4, "dealer": "Ada"},
    {"hand_number": 5, "dealer": "Linus"},
    {"hand_number": 6, "dealer": "Guido"},
    {"hand_number": 7, "dealer": "Ada"},
    {"hand_number": 8, "dealer": "Linus"},
])

print("Problem 16 solution passed.")


Problem 16 solution passed.


# Problem 17 — Infinite grid coordinates

Implement `grid_coordinates(width)`.

It should lazily yield row/column pairs in row-major order:

```python
take(8, grid_coordinates(3))
# [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1)]
```

Requirements:

- Use `count`.
- Validate `width > 0`.
- The iterator must be infinite.


In [21]:
# Solution 17

def grid_coordinates(width):
    if width <= 0:
        raise ValueError("width must be positive")

    for index in count():
        yield divmod(index, width)

assert_equal(
    take(8, grid_coordinates(3)),
    [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1)]
)

assert_equal(nth(grid_coordinates(10), 123), (12, 3))

print("Problem 17 solution passed.")


Problem 17 solution passed.


# Problem 18 — Batch labels using `repeat` and `count`

You receive batches of items. For each batch, you want labels like:

```python
batch-001-item-000
batch-001-item-001
...
batch-002-item-000
```

Implement `label_batches(batch_sizes)`.

Requirements:

- Use `count(1)` for batch numbers.
- Use `count(0)` for item numbers inside each batch.
- Use `repeat` to repeat the current batch label for all items in that batch.
- Return a flat list of labels.


In [22]:
# Solution 18

def label_batches(batch_sizes):
    labels = []

    for batch_number, batch_size in zip(count(1), batch_sizes):
        if batch_size < 0:
            raise ValueError("batch sizes must be non-negative")

        batch_label = f"batch-{batch_number:03d}"

        for current_batch_label, item_number in zip(
            repeat(batch_label, batch_size),
            count(0),
        ):
            labels.append(f"{current_batch_label}-item-{item_number:03d}")

    return labels

assert_equal(label_batches([3, 0, 2]), [
    "batch-001-item-000",
    "batch-001-item-001",
    "batch-001-item-002",
    "batch-003-item-000",
    "batch-003-item-001",
])

print("Problem 18 solution passed.")


Problem 18 solution passed.


# Problem 19 — Lazy simulation of a traffic light

Implement `traffic_light_events(start, seconds_per_state, states)`.

Each event should contain:

```python
{
    "event_id": 1,
    "timestamp": datetime(...),
    "state": "red"
}
```

Requirements:

- Use `count` for event IDs.
- Use `datetime_count` for timestamps.
- Use `cycle` for states.
- Return an infinite iterator.


In [23]:
# Solution 19

def traffic_light_events(start, seconds_per_state, states=("red", "green", "yellow")):
    if seconds_per_state <= 0:
        raise ValueError("seconds_per_state must be positive")
    if not states:
        raise ValueError("states must not be empty")

    timestamps = datetime_count(start, timedelta(seconds=seconds_per_state))

    for event_id, timestamp, state in zip(count(1), timestamps, cycle(states)):
        yield {
            "event_id": event_id,
            "timestamp": timestamp,
            "state": state,
        }

events = take(7, traffic_light_events(datetime(2026, 1, 1, 8, 0), 30))
assert_equal([e["event_id"] for e in events], [1, 2, 3, 4, 5, 6, 7])
assert_equal([e["state"] for e in events], ["red", "green", "yellow", "red", "green", "yellow", "red"])
assert_equal(events[3]["timestamp"], datetime(2026, 1, 1, 8, 1, 30))

print("Problem 19 solution passed.")


Problem 19 solution passed.


# Problem 20 — Constant arguments with `repeat` and `starmap`

You have a function:

```python
pow(base, exponent)
```

Use `repeat` and `starmap` to compute powers of a constant base for many exponents.

Implement `powers_of(base, exponents)`.

Requirements:

- Use `repeat(base)`.
- Use `zip(repeat(base), exponents)`.
- Use `starmap(pow, ...)`.
- Stop when `exponents` stops.


In [24]:
# Solution 20

def powers_of(base, exponents):
    argument_pairs = zip(repeat(base), exponents)
    return starmap(pow, argument_pairs)

assert_equal(list(powers_of(2, range(8))), [1, 2, 4, 8, 16, 32, 64, 128])
assert_equal(list(powers_of(10, [0, 1, 2, 3])), [1, 10, 100, 1000])

# The repeated base is infinite, but the zip stops when exponents stops.
assert_equal(list(powers_of(3, [])), [])

print("Problem 20 solution passed.")


Problem 20 solution passed.


# Problem 21 — Randomized testing for `deal`

Write a small randomized test for `deal`.

Requirements:

- Test several player counts from 1 through 10.
- Test that all 52 cards are dealt.
- Test that no card is duplicated.
- Test that hand sizes differ by at most 1.


In [25]:
# Solution 21

def test_deal_randomized():
    deck = list(card_deck())

    for number_of_players in range(1, 11):
        players = [f"player-{i}" for i in range(number_of_players)]
        random.shuffle(deck)

        hands = deal(deck, players)

        all_cards = [card for hand in hands.values() for card in hand]
        hand_sizes = [len(hand) for hand in hands.values()]

        assert_equal(len(all_cards), 52)
        assert_equal(len(set(all_cards)), 52)
        assert max(hand_sizes) - min(hand_sizes) <= 1

test_deal_randomized()

print("Problem 21 solution passed.")


Problem 21 solution passed.


# Problem 22 — Detect repetition frequency in a cycled weighted stream

Given a finite sample from a weighted cycle, estimate the ratio of labels.

Use the `weighted_cycle` function from Problem 8.

Requirements:

- Create a stream with weights `{"gold": 1, "silver": 2, "bronze": 5}`.
- Take 800 items.
- Count each label.
- Verify the observed counts match the expected ratio exactly for 800 items.


In [26]:
# Solution 22

weights = {"gold": 1, "silver": 2, "bronze": 5}
sample = take(800, weighted_cycle(weights))
counts = Counter(sample)

# Pattern length is 8, and 800 is a multiple of 8, so the ratio is exact.
assert_equal(counts, Counter({"gold": 100, "silver": 200, "bronze": 500}))

print(counts)
print("Problem 22 solution passed.")


Counter({'bronze': 500, 'silver': 200, 'gold': 100})
Problem 22 solution passed.


# Problem 23 — Create a lazy repeating window of recent values

Implement `cyclic_windows(pattern, window_size)`.

Example:

```python
take(5, cyclic_windows("ABC", 4))
# [("A", "B", "C", "A"), ("B", "C", "A", "B"), ...]
```

Requirements:

- Use `cycle`.
- Return an infinite iterator of tuples.
- Validate `pattern` is not empty.
- Validate `window_size > 0`.


In [27]:
# Solution 23

def cyclic_windows(pattern, window_size):
    pattern = tuple(pattern)

    if not pattern:
        raise ValueError("pattern must not be empty")
    if window_size <= 0:
        raise ValueError("window_size must be positive")

    stream = cycle(pattern)
    window = deque(take(window_size, stream), maxlen=window_size)

    while True:
        yield tuple(window)
        window.append(next(stream))

assert_equal(
    take(5, cyclic_windows("ABC", 4)),
    [
        ("A", "B", "C", "A"),
        ("B", "C", "A", "B"),
        ("C", "A", "B", "C"),
        ("A", "B", "C", "A"),
        ("B", "C", "A", "B"),
    ]
)

print("Problem 23 solution passed.")


Problem 23 solution passed.


# Problem 24 — Build a reusable infinite ticker

Create a class `Ticker`.

Requirements:

- Constructor arguments: `symbol`, `start_price`, `step`.
- Use `count(start_price, step)` to generate prices.
- Each item should be a dictionary with `tick`, `symbol`, and `price`.
- `tick` should start at 1.
- Provide a method `.take(n)` that returns the next `n` events.


In [28]:
# Solution 24

class Ticker:
    def __init__(self, symbol, start_price, step):
        self.symbol = symbol
        self._ticks = count(1)
        self._prices = count(start_price, step)

    def __iter__(self):
        return self

    def __next__(self):
        return {
            "tick": next(self._ticks),
            "symbol": self.symbol,
            "price": next(self._prices),
        }

    def take(self, n):
        return take(n, self)

ticker = Ticker("PY", Decimal("100.00"), Decimal("0.25"))
events = ticker.take(4)

assert_equal(events, [
    {"tick": 1, "symbol": "PY", "price": Decimal("100.00")},
    {"tick": 2, "symbol": "PY", "price": Decimal("100.25")},
    {"tick": 3, "symbol": "PY", "price": Decimal("100.50")},
    {"tick": 4, "symbol": "PY", "price": Decimal("100.75")},
])

more = ticker.take(2)
assert_equal([event["tick"] for event in more], [5, 6])

print("Problem 24 solution passed.")


Problem 24 solution passed.


# Final best-practice checklist

Before using infinite iterators in production code, ask:

1. What bounds the iteration?
2. Could `cycle` cache too much?
3. Could `repeat` accidentally reuse a mutable object?
4. Do I need exact arithmetic?
5. Do I have tests for empty inputs, zero steps, negative values, and uneven lengths?
6. Is my generator lazy, or did I accidentally materialize a huge collection?


In [29]:
# Final quick run: a compact demonstration combining all three infinite iterators.

def compact_demo():
    users = ["Ada", "Linus", "Guido"]
    timestamps = datetime_count(datetime(2026, 1, 1, 12, 0), timedelta(minutes=5))

    return take(6, (
        {
            "id": event_id,
            "time": timestamp,
            "user": user,
            "kind": kind,
        }
        for event_id, timestamp, user, kind in zip(
            count(1),
            timestamps,
            cycle(users),
            repeat("heartbeat"),
        )
    ))

demo = compact_demo()
assert_equal([row["id"] for row in demo], [1, 2, 3, 4, 5, 6])
assert_equal([row["user"] for row in demo], ["Ada", "Linus", "Guido", "Ada", "Linus", "Guido"])
assert_equal(set(row["kind"] for row in demo), {"heartbeat"})

demo


[{'id': 1,
  'time': datetime.datetime(2026, 1, 1, 12, 0),
  'user': 'Ada',
  'kind': 'heartbeat'},
 {'id': 2,
  'time': datetime.datetime(2026, 1, 1, 12, 5),
  'user': 'Linus',
  'kind': 'heartbeat'},
 {'id': 3,
  'time': datetime.datetime(2026, 1, 1, 12, 10),
  'user': 'Guido',
  'kind': 'heartbeat'},
 {'id': 4,
  'time': datetime.datetime(2026, 1, 1, 12, 15),
  'user': 'Ada',
  'kind': 'heartbeat'},
 {'id': 5,
  'time': datetime.datetime(2026, 1, 1, 12, 20),
  'user': 'Linus',
  'kind': 'heartbeat'},
 {'id': 6,
  'time': datetime.datetime(2026, 1, 1, 12, 25),
  'user': 'Guido',
  'kind': 'heartbeat'}]